## Download triangle mesh from Evo

This notebook downloads one or more triangle-mesh objects from an Evo workspace.

In Evo, geological model output volumes are stored as `triangle-mesh` geoscience objects. The notebook downloads the object definition and its referenced parquet tables, then writes local CSV and OBJ files.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

Provide these application settings before running the setup cell:
- The client ID of your Evo application.
- The callback or redirect URL of your Evo application.

### Files written locally

Each downloaded triangle mesh is exported into its own local folder containing:
- `object.json`
- `vertices.csv`
- `triangles.csv`
- `edges.csv` when the mesh includes edges
- `mesh.obj`

In [ ]:
import json
import re
from pathlib import Path
from uuid import UUID

import pandas as pd
from IPython.display import display

from evo.notebooks import FeedbackWidget, ServiceManagerWidget
from evo.objects import ObjectAPIClient

# Evo app credentials
client_id = "<your-client-id>"
# redirect_url = "<your-redirect-url>"

manager = await ServiceManagerWidget.with_auth_code(
    # redirect_url=redirect_url,
    client_id=client_id,
    cache_location="./notebook-data",
).login()

object_client = ObjectAPIClient(
    manager.get_environment(),
    manager.get_connector(),
    cache=manager.cache,
)

### List triangle-mesh objects in the workspace

Run the next cell to list the triangle-mesh objects in the selected workspace. These are the output volumes you can export.

In [ ]:
all_objects = await object_client.list_all_objects()

output_volume_index = pd.DataFrame(
    [
        {
            "name": obj.name,
            "path": obj.path,
            "id": str(obj.id),
            "version_id": obj.version_id,
        }
        for obj in all_objects
        if obj.schema_id.sub_classification == "triangle-mesh"
    ]
)

if output_volume_index.empty:
    print("No triangle-mesh objects were found in this workspace.")
else:
    output_volume_index = output_volume_index.sort_values(["path", "name"]).reset_index(drop=True)
    display(output_volume_index)

### Select output volumes and define helpers

Add one or more object IDs or object paths in the next cell. The helper functions download the mesh tables and export each selected output volume to a local folder under `downloads/output-volumes`.

In [ ]:
selected_object_ids = [
    "070365ec-7f45-4bf1-8abb-2fde66c3ae06",
]

selected_object_paths = [
    # "/geological-models/example-output-volume.json",
]

download_root = Path("downloads/output-volumes")


def sanitize_name(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", value).strip(".-") or "output-volume"


def build_output_dir(root, downloaded_object):
    folder_name = f"{sanitize_name(downloaded_object.metadata.name)}-{downloaded_object.metadata.id}"
    target = root / folder_name
    target.mkdir(parents=True, exist_ok=True)
    return target


def write_obj(vertices, triangles, destination):
    with destination.open("w", encoding="utf-8") as handle:
        for row in vertices.itertuples(index=False):
            handle.write(f"v {row.x} {row.y} {row.z}\n")

        for row in triangles.itertuples(index=False):
            handle.write(f"f {int(row.i) + 1} {int(row.j) + 1} {int(row.k) + 1}\n")


async def download_output_volume(downloaded_object, output_root):
    if downloaded_object.metadata.schema_id.sub_classification != "triangle-mesh":
        raise ValueError(
            f"Expected a triangle-mesh object, got '{downloaded_object.metadata.schema_id.sub_classification}'."
        )

    output_dir = build_output_dir(output_root, downloaded_object)
    mesh_dict = downloaded_object.as_dict()

    vertices = await downloaded_object.download_dataframe(
        "triangles.vertices",
        column_names=["x", "y", "z"],
        fb=FeedbackWidget(f"Downloading vertices for '{downloaded_object.metadata.name}'"),
    )
    triangles = await downloaded_object.download_dataframe(
        "triangles.indices",
        column_names=["i", "j", "k"],
        fb=FeedbackWidget(f"Downloading triangles for '{downloaded_object.metadata.name}'"),
    )

    edges_info = downloaded_object.search("edges.indices")
    if edges_info is not None:
        edges = await downloaded_object.download_dataframe(
            "edges.indices",
            column_names=["start", "end"],
            fb=FeedbackWidget(f"Downloading edges for '{downloaded_object.metadata.name}'"),
        )
    else:
        edges = None

    with (output_dir / "object.json").open("w", encoding="utf-8") as handle:
        json.dump(mesh_dict, handle, indent=2, default=str)

    vertices.to_csv(output_dir / "vertices.csv", index=False)
    triangles.to_csv(output_dir / "triangles.csv", index=False)
    if edges is not None:
        edges.to_csv(output_dir / "edges.csv", index=False)

    write_obj(vertices, triangles, output_dir / "mesh.obj")

    return {
        "name": downloaded_object.metadata.name,
        "path": downloaded_object.metadata.path,
        "id": str(downloaded_object.metadata.id),
        "version_id": downloaded_object.metadata.version_id,
        "vertices": len(vertices),
        "triangles": len(triangles),
        "edges": 0 if edges is None else len(edges),
        "output_dir": str(output_dir),
    }


async def download_output_volume_by_id(object_id, output_root):
    downloaded_object = await object_client.download_object_by_id(UUID(object_id))
    return await download_output_volume(downloaded_object, output_root)


async def download_output_volume_by_path(object_path, output_root):
    downloaded_object = await object_client.download_object_by_path(path=object_path)
    return await download_output_volume(downloaded_object, output_root)

### Download the selected output volumes

After you have filled in one or more object IDs or object paths, run the next cell to export the selected output volumes.

In [ ]:
if not selected_object_ids and not selected_object_paths:
    raise ValueError("Add at least one object ID or object path before running this cell.")

download_root.mkdir(parents=True, exist_ok=True)
download_summaries = []

for object_id in selected_object_ids:
    download_summaries.append(await download_output_volume_by_id(object_id, download_root))

# for object_path in selected_object_paths:
#     download_summaries.append(await download_output_volume_by_path(object_path, download_root))

download_results = pd.DataFrame(download_summaries)
display(download_results)

### Inspect the generated local files

Run the next cell to show the files written for the first downloaded output volume.

In [ ]:
if download_results.empty:
    raise ValueError("No output volumes have been downloaded yet. Run the previous code cell first.")

first_output_dir = Path(download_results.iloc[0]["output_dir"])
pd.DataFrame({"file": sorted(path.name for path in first_output_dir.iterdir())})